In [2]:
%run "01_Data Loading.ipynb"

All imports successful
Train shape : (2666, 20)
Test  shape : (667, 20)

Column dtypes:
State                      object
Account length              int64
Area code                   int64
International plan         object
Voice mail plan            object
Number vmail messages       int64
Total day minutes         float64
Total day calls             int64
Total day charge          float64
Total eve minutes         float64
Total eve calls             int64
Total eve charge          float64
Total night minutes       float64
Total night calls           int64
Total night charge        float64
Total intl minutes        float64
Total intl calls            int64
Total intl charge         float64
Customer service calls      int64
Churn                        bool
dtype: object

Missing values: 0


In [10]:
def preprocess(df, le_dict=None, fit=True):
    """Encode, drop non-predictive columns, and engineer new features."""
    df = df.copy()

    df.drop(columns=[c for c in ['State', 'Area code'] if c in df.columns], inplace=True)

    df['Churn'] = df['Churn'].astype(int)

    for col in ['International plan', 'Voice mail plan']:
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            if le_dict is not None: le_dict[col] = le
        else:
            df[col] = le_dict[col].transform(df[col])

    df['Total minutes'] = df[['Total day minutes', 'Total eve minutes',
                               'Total night minutes', 'Total intl minutes']].sum(axis=1)
    df['Total charges'] = df[['Total day charge', 'Total eve charge',
                               'Total night charge', 'Total intl charge']].sum(axis=1)
    df['Day usage pct'] = df['Total day minutes'] / (df['Total minutes'] + 1e-9)
    df['CS call flag']  = (df['Customer service calls'] >= 4).astype(int)

    charge_cols = ['Total day charge', 'Total eve charge',
                   'Total night charge', 'Total intl charge']
    df.drop(columns=[c for c in charge_cols if c in df.columns], inplace=True)

    return df

le_dict  = {}
train_p  = preprocess(train.copy(), le_dict, fit=True)
test_p   = preprocess(test.copy(),  le_dict, fit=False)

X_train, y_train = train_p.drop(columns=['Churn']), train_p['Churn']
X_test,  y_test  = test_p.drop(columns=['Churn']),  test_p['Churn']

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"Features: {list(X_train.columns)}")

X_train: (2666, 17)  |  X_test: (667, 17)
Features: ['Account length', 'International plan', 'Voice mail plan', 'Number vmail messages', 'Total day minutes', 'Total day calls', 'Total eve minutes', 'Total eve calls', 'Total night minutes', 'Total night calls', 'Total intl minutes', 'Total intl calls', 'Customer service calls', 'Total minutes', 'Total charges', 'Day usage pct', 'CS call flag']
